In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os


In [8]:
project_path = os.path.dirname(os.getcwd())

data_path = os.path.join(project_path, "ipl_male_json")

print("Dataset path:", data_path)
print("Folder exists:", os.path.exists(data_path))

Dataset path: C:\Users\sw\Desktop\Fruti Project\BoundaryMetrics\ipl_male_json
Folder exists: True


In [9]:
files = [
    file for file in os.listdir(data_path)
    if file.endswith(".json")
]

print("Total IPL matches:", len(files))

Total IPL matches: 1243


In [10]:
file_path = os.path.join(data_path, files[0])

with open(file_path, "r") as file:
    match_data = json.load(file)

match_data.keys()

dict_keys(['meta', 'info', 'innings'])

In [13]:
# Checking the basic details of this match
info = match_data["info"]

print("Season:", info["season"])
print("Teams:", info["teams"])
print("Venue:", info.get("venue"))
print("City:", info.get("city"))
print("Date:", info["dates"])
print("Toss:", info["toss"])

Season: 2017
Teams: ['Sunrisers Hyderabad', 'Royal Challengers Bangalore']
Venue: Rajiv Gandhi International Stadium, Uppal
City: Hyderabad
Date: ['2017-04-05']
Toss: {'decision': 'field', 'winner': 'Royal Challengers Bangalore'}


In [14]:
# Checking how many innings are available in this match
innings = match_data["innings"]

print("Total innings:", len(innings))

Total innings: 2


In [15]:
# Taking a look at the deliveries recorded in the first over
innings[0]["overs"][0]

{'over': 0,
 'deliveries': [{'actual_delivery': '0.1',
   'batter': 'DA Warner',
   'bowler': 'TS Mills',
   'non_striker': 'S Dhawan',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'actual_delivery': '0.2',
   'batter': 'DA Warner',
   'bowler': 'TS Mills',
   'non_striker': 'S Dhawan',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'actual_delivery': '0.3',
   'batter': 'DA Warner',
   'bowler': 'TS Mills',
   'non_striker': 'S Dhawan',
   'runs': {'batter': 4, 'extras': 0, 'total': 4}},
  {'actual_delivery': '0.4',
   'batter': 'DA Warner',
   'bowler': 'TS Mills',
   'non_striker': 'S Dhawan',
   'runs': {'batter': 0, 'extras': 0, 'total': 0}},
  {'actual_delivery': '0.5',
   'batter': 'DA Warner',
   'bowler': 'TS Mills',
   'extras': {'wides': 2},
   'non_striker': 'S Dhawan',
   'runs': {'batter': 0, 'extras': 2, 'total': 2}},
  {'actual_delivery': '0.5',
   'batter': 'S Dhawan',
   'bowler': 'TS Mills',
   'non_striker': 'DA Warner',
   'runs': {'batter': 0, 'e

In [18]:
# Collecting deliveries from all IPL matches
deliveries = []

for file_name in files:
    file_path = os.path.join(data_path, file_name)

    with open(file_path, "r") as file:
        match_data = json.load(file)

    match_id = file_name.replace(".json", "")

    for inning_number, inning in enumerate(match_data["innings"], start=1):

        batting_team = inning["team"]

        for over_data in inning["overs"]:
            over_number = over_data["over"]

            for delivery_number, delivery in enumerate(over_data["deliveries"], start=1):

                deliveries.append({
                    "match_id": match_id,
                    "innings": inning_number,
                    "over": over_number,
                    "delivery_number": delivery_number,
                    "batting_team": batting_team,
                    "batter": delivery["batter"],
                    "bowler": delivery["bowler"],
                    "runs_batter": delivery["runs"]["batter"],
                    "runs_total": delivery["runs"]["total"],
                    "extras": delivery.get("extras", {}),
                    "wickets": delivery.get("wickets", [])
                })

df = pd.DataFrame(deliveries)
df.head()

,match_id,innings,over,delivery_number,batting_team,batter,bowler,runs_batter,runs_total,extras,wickets
0,1082591,1,0,1,Sunrisers Hyderabad,DA Warner,TS Mills,0,0,{},[]
1,1082591,1,0,2,Sunrisers Hyderabad,DA Warner,TS Mills,0,0,{},[]
2,1082591,1,0,3,Sunrisers Hyderabad,DA Warner,TS Mills,4,4,{},[]
3,1082591,1,0,4,Sunrisers Hyderabad,DA Warner,TS Mills,0,0,{},[]
4,1082591,1,0,5,Sunrisers Hyderabad,DA Warner,TS Mills,0,2,{'wides': 2},[]


In [19]:
print("Total deliveries:", len(df))
print("Total columns:", len(df.columns))

Total deliveries: 295732
Total columns: 11


In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 295732 entries, 0 to 295731
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   match_id         295732 non-null  str   
 1   innings          295732 non-null  int64 
 2   over             295732 non-null  int64 
 3   delivery_number  295732 non-null  int64 
 4   batting_team     295732 non-null  str   
 5   batter           295732 non-null  str   
 6   bowler           295732 non-null  str   
 7   runs_batter      295732 non-null  int64 
 8   runs_total       295732 non-null  int64 
 9   extras           295732 non-null  object
 10  wickets          295732 non-null  object
dtypes: int64(5), object(2), str(4)
memory usage: 24.8+ MB


In [21]:
# Checking missing values and duplicate deliveries
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate deliveries:")
print(df.duplicated(
    subset=["match_id", "innings", "over", "delivery_number"]
).sum())

Missing values:
match_id           0
innings            0
over               0
delivery_number    0
batting_team       0
batter             0
bowler             0
runs_batter        0
runs_total         0
extras             0
wickets            0
dtype: int64

Duplicate deliveries:
0


In [22]:
# Separating extras so they can be used in calculations
df["wides"] = df["extras"].apply(lambda x: x.get("wides", 0))
df["noballs"] = df["extras"].apply(lambda x: x.get("noballs", 0))
df["byes"] = df["extras"].apply(lambda x: x.get("byes", 0))
df["legbyes"] = df["extras"].apply(lambda x: x.get("legbyes", 0))
df["penalty"] = df["extras"].apply(lambda x: x.get("penalty", 0))

# Wides and no-balls do not count as legal deliveries
df["is_legal_delivery"] = (
    (df["wides"] == 0) & (df["noballs"] == 0)
).astype(int)

# Removing byes, leg-byes and penalty runs from the bowler's conceded runs
df["bowler_runs"] = (
    df["runs_total"] - df["byes"] - df["legbyes"] - df["penalty"]
)

# A batter faces a delivery unless it is a wide
df["balls_faced"] = (df["wides"] == 0).astype(int)

df[[
    "runs_total", "wides", "noballs", "byes",
    "legbyes", "is_legal_delivery", "bowler_runs", "balls_faced"
]].head(10)

,runs_total,wides,noballs,byes,legbyes,is_legal_delivery,bowler_runs,balls_faced
0,0,0,0,0,0,1,0,1
1,0,0,0,0,0,1,0,1
2,4,0,0,0,0,1,4,1
3,0,0,0,0,0,1,0,1
4,2,2,0,0,0,0,2,0
5,0,0,0,0,0,1,0,1
6,1,0,0,0,1,1,0,1
7,1,0,0,0,0,1,1,1
8,4,0,0,0,0,1,4,1
9,1,0,1,0,0,0,1,1


In [23]:
# Counting wickets and keeping their dismissal details
df["total_wickets"] = df["wickets"].apply(len)

df["wicket_types"] = df["wickets"].apply(
    lambda x: [wicket["kind"] for wicket in x]
)

df["players_dismissed"] = df["wickets"].apply(
    lambda x: [wicket["player_out"] for wicket in x]
)

# Only these dismissals are credited to the bowler
bowler_wicket_types = [
    "bowled", "caught", "caught and bowled",
    "lbw", "stumped", "hit wicket"
]

df["bowler_wickets"] = df["wicket_types"].apply(
    lambda types: sum(
        wicket_type in bowler_wicket_types
        for wicket_type in types
    )
)

df[[
    "wickets", "total_wickets", "wicket_types",
    "players_dismissed", "bowler_wickets"
]].head(10)

,wickets,total_wickets,wicket_types,players_dismissed,bowler_wickets
0,[],0,[],[],0
1,[],0,[],[],0
2,[],0,[],[],0
3,[],0,[],[],0
4,[],0,[],[],0
5,[],0,[],[],0
6,[],0,[],[],0
7,[],0,[],[],0
8,[],0,[],[],0
9,[],0,[],[],0


In [24]:
# Collecting basic details from every match
match_details = []

for file_name in files:
    file_path = os.path.join(data_path, file_name)

    with open(file_path, "r") as file:
        match_data = json.load(file)

    info = match_data["info"]

    match_details.append({
        "match_id": file_name.replace(".json", ""),
        "date": info["dates"][0],
        "season": info["season"],
        "venue": info.get("venue"),
        "city": info.get("city"),
        "team1": info["teams"][0],
        "team2": info["teams"][1],
        "toss_winner": info["toss"]["winner"],
        "toss_decision": info["toss"]["decision"],
        "winner": info.get("outcome", {}).get("winner")
    })

matches_df = pd.DataFrame(match_details)
matches_df["date"] = pd.to_datetime(matches_df["date"])

matches_df.head()

,match_id,date,season,venue,city,team1,team2,toss_winner,toss_decision,winner
0,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,Sunrisers Hyderabad
1,1082592,2017-04-06,2017,"Maharashtra Cricket Association Stadium, Pune",Pune,Rising Pune Supergiant,Mumbai Indians,Rising Pune Supergiant,field,Rising Pune Supergiant
2,1082593,2017-04-07,2017,"Saurashtra Cricket Association Stadium, Rajkot",Rajkot,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,Kolkata Knight Riders
3,1082594,2017-04-08,2017,"Holkar Cricket Stadium, Indore",Indore,Kings XI Punjab,Rising Pune Supergiant,Kings XI Punjab,field,Kings XI Punjab
4,1082595,2017-04-08,2017,"M Chinnaswamy Stadium, Bengaluru",Bengaluru,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,Royal Challengers Bangalore


In [25]:
# Connecting delivery records with their match details
df = df.merge(
    matches_df[["match_id", "date", "season", "venue", "city"]],
    on="match_id",
    how="left"
)

df[["match_id", "date", "season", "venue", "city"]].head()

,match_id,date,season,venue,city
0,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
1,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
2,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
3,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
4,1082591,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad


In [26]:
# Removing complex columns after extracting the information we need
df_clean = df.drop(
    columns=[
        "extras", "wickets",
        "wicket_types", "players_dismissed"
    ]
).copy()

# Keeping deliveries in match order
df_clean = df_clean.sort_values(
    ["match_id", "innings", "over", "delivery_number"]
).reset_index(drop=True)

df_clean.head()

,match_id,innings,over,delivery_number,batting_team,batter,bowler,runs_batter,runs_total,wides,...,penalty,is_legal_delivery,bowler_runs,balls_faced,total_wickets,bowler_wickets,date,season,venue,city
0,1082591,1,0,1,Sunrisers Hyderabad,DA Warner,TS Mills,0,0,0,...,0,1,0,1,0,0,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
1,1082591,1,0,2,Sunrisers Hyderabad,DA Warner,TS Mills,0,0,0,...,0,1,0,1,0,0,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
2,1082591,1,0,3,Sunrisers Hyderabad,DA Warner,TS Mills,4,4,0,...,0,1,4,1,0,0,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
3,1082591,1,0,4,Sunrisers Hyderabad,DA Warner,TS Mills,0,0,0,...,0,1,0,1,0,0,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad
4,1082591,1,0,5,Sunrisers Hyderabad,DA Warner,TS Mills,0,2,2,...,0,0,2,0,0,0,2017-04-05,2017,"Rajiv Gandhi International Stadium, Uppal",Hyderabad


In [27]:
print("Dataset shape:", df_clean.shape)

print("\nColumn information:")
df_clean.info()

print("\nMissing values:")
print(df_clean.isnull().sum())

Dataset shape: (295732, 23)

Column information:
<class 'pandas.DataFrame'>
RangeIndex: 295732 entries, 0 to 295731
Data columns (total 23 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   match_id           295732 non-null  str           
 1   innings            295732 non-null  int64         
 2   over               295732 non-null  int64         
 3   delivery_number    295732 non-null  int64         
 4   batting_team       295732 non-null  str           
 5   batter             295732 non-null  str           
 6   bowler             295732 non-null  str           
 7   runs_batter        295732 non-null  int64         
 8   runs_total         295732 non-null  int64         
 9   wides              295732 non-null  int64         
 10  noballs            295732 non-null  int64         
 11  byes               295732 non-null  int64         
 12  legbyes            295732 non-null  int64         
 13  penalt

In [28]:
# Counting every recorded dismissal
dismissal_stats = (
    df["wicket_types"]
    .explode()
    .dropna()
    .value_counts()
    .reset_index()
)

dismissal_stats.columns = ["dismissal_type", "count"]

dismissal_stats

,dismissal_type,count
0,caught,9321
1,bowled,2460
2,run out,1194
3,lbw,883
4,caught and bowled,410
5,stumped,388
6,hit wicket,20
7,retired hurt,19
8,retired out,6
9,obstructing the field,4


In [29]:
# Creating the output folder if it doesn't already exist
output_path = os.path.join(project_path, "data")
os.makedirs(output_path, exist_ok=True)

# Saving the datasets for the next analysis notebooks
df_clean.to_csv(
    os.path.join(output_path, "deliveries_clean.csv"),
    index=False
)

matches_df.to_csv(
    os.path.join(output_path, "matches.csv"),
    index=False
)

dismissal_stats.to_csv(
    os.path.join(output_path, "dismissal_stats.csv"),
    index=False
)

print("All datasets saved successfully!")

All datasets saved successfully!


In [30]:
# Checking whether the cleaned dataset contains valid records
print("Total deliveries:", len(df_clean))
print("Unique matches:", df_clean["match_id"].nunique())
print("Unique batters:", df_clean["batter"].nunique())
print("Unique bowlers:", df_clean["bowler"].nunique())

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate deliveries:")
print(df_clean.duplicated(
    subset=["match_id", "innings", "over", "delivery_number"]
).sum())

Total deliveries: 295732
Unique matches: 1243
Unique batters: 738
Unique bowlers: 577

Missing values:
match_id                 0
innings                  0
over                     0
delivery_number          0
batting_team             0
batter                   0
bowler                   0
runs_batter              0
runs_total               0
wides                    0
noballs                  0
byes                     0
legbyes                  0
penalty                  0
is_legal_delivery        0
bowler_runs              0
balls_faced              0
total_wickets            0
bowler_wickets           0
date                     0
season                   0
venue                    0
city                 12397
dtype: int64

Duplicate deliveries:
0


In [31]:
# Checking whether total runs are consistent with their components
print("Total runs:", df_clean["runs_total"].sum())

print(
    "Batter runs:",
    df_clean["runs_batter"].sum()
)

print(
    "Extra runs:",
    (
        df_clean["runs_total"] - df_clean["runs_batter"]
    ).sum()
)

print(
    "Bowler runs:",
    df_clean["bowler_runs"].sum()
)

Total runs: 401738
Batter runs: 381529
Extra runs: 20209
Bowler runs: 394585


In [32]:
# Checking runs that are not charged to bowlers
excluded_runs = (
    df_clean["byes"].sum()
    + df_clean["legbyes"].sum()
    + df_clean["penalty"].sum()
)

print("Excluded runs:", excluded_runs)
print("Difference in total and bowler runs:",
      df_clean["runs_total"].sum() - df_clean["bowler_runs"].sum())

Excluded runs: 7153
Difference in total and bowler runs: 7153


In [33]:
print(matches_df.columns.tolist())
print(df_clean.columns.tolist())

['match_id', 'date', 'season', 'venue', 'city', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner']
['match_id', 'innings', 'over', 'delivery_number', 'batting_team', 'batter', 'bowler', 'runs_batter', 'runs_total', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'is_legal_delivery', 'bowler_runs', 'balls_faced', 'total_wickets', 'bowler_wickets', 'date', 'season', 'venue', 'city']


In [34]:
# Prepare deliveries data for MySQL import
mysql_deliveries = df_clean.rename(columns={
    "over": "over_number"
})

mysql_deliveries = mysql_deliveries[
    [
        "match_id", "innings", "over_number", "delivery_number",
        "batting_team", "batter", "bowler",
        "runs_batter", "runs_total", "wides", "noballs",
        "byes", "legbyes", "penalty", "is_legal_delivery",
        "bowler_runs", "balls_faced", "total_wickets", "bowler_wickets"
    ]
]

mysql_deliveries.to_csv(
    os.path.join(output_path, "deliveries_mysql.csv"),
    index=False
)

print("MySQL deliveries CSV created:", mysql_deliveries.shape)

MySQL deliveries CSV created: (295732, 19)
